# 07 - Análisis regional de ingresos ENIGH 2018-2024

Este notebook inicia la etapa regional del proyecto. El objetivo es usar los marts ya construidos en `data/interim/revision_4` para explorar cómo cambia el ingreso laboral por región Banxico, entidad, tamaño de localidad, estrato socioeconómico, educación, sexo y variables laborales.

La prioridad de esta libreta es interpretabilidad y reproducibilidad para la tesina: no se estiman modelos, no se reconstruye ingreso y no se hacen inferencias complejas con diseño muestral. Se usa el `factor` solo para medias descriptivas ponderadas cuando está disponible.

In [ ]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown
from matplotlib.ticker import FuncFormatter

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({
    "figure.figsize": (10, 5),
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
})

def find_project_root():
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data" / "interim" / "revision_4").exists():
            return candidate
    raise FileNotFoundError("No se encontró data/interim/revision_4 desde el directorio actual.")

ROOT = find_project_root()
REV4 = ROOT / "data" / "interim" / "revision_4"
PERSONA_PATH = REV4 / "mart_persona_2018_2024.csv.gz"
HOGAR_PATH = REV4 / "mart_hogar_2018_2024.csv.gz"
REPORT_PATH = ROOT / "reports" / "revisión 4.md"
ESTADO_ARTE_PATH = ROOT / "reports" / "estado_del_arte_geografia_ingresos_ENIGH.md"

TARGET_MAIN = "ingreso_persona_laboral_negocio_tri"
TARGET_COMPLEMENT_PERSONA = "ing_cor_hogar_pc_oficial_tri"
TARGET_COMPLEMENT_HOGAR = "ing_cor_pc_oficial_tri"
WEIGHT = "factor"
REGION_ORDER = ["Norte", "Centro Norte", "Centro", "Sur"]

print(f"Proyecto: {ROOT}")
print(f"Mart persona: {PERSONA_PATH.exists()} | Mart hogar: {HOGAR_PATH.exists()}")
print(f"Target principal: {TARGET_MAIN}")

## Carga de datos

Se cargan solo las columnas necesarias para el análisis regional. El mart completo permanece en disco; esta selección evita consumir memoria en variables que no se usan en esta etapa.

In [ ]:
PERSONA_COLS = [
    "anio", "folioviv", "foliohog", "numren",
    "region_banxico", "cve_ent", "entidad", "cve_mun", "municipio",
    "tam_loc", "tam_loc_desc", "est_socio", "est_socio_desc",
    "factor", "factor_hogar", "est_dis", "upm",
    "sexo_desc", "edad", "nivelaprob_desc", "nivel_desc",
    "tiene_trabajo_reportado", "num_trabaj_desc",
    "contrato_principal_desc", "subor_principal_desc", "pago_principal_desc",
    "tiene_suel_principal_desc", "tam_emp_principal_desc",
    TARGET_MAIN, "ingreso_persona_total_registros_tri", TARGET_COMPLEMENT_PERSONA,
    "ingtrab_hogar_pc_oficial_tri",
]

HOGAR_COLS = [
    "anio", "folioviv", "foliohog",
    "region_banxico", "cve_ent", "entidad", "cve_mun", "municipio",
    "tam_loc", "tam_loc_desc", "est_socio", "est_socio_desc",
    "factor", "factor_hogar", "est_dis", "upm",
    TARGET_COMPLEMENT_HOGAR, "ingtrab_pc_oficial_tri",
]

def read_existing(path, columns):
    available = pd.read_csv(path, nrows=0).columns.tolist()
    missing = [c for c in columns if c not in available]
    if missing:
        print(f"Columnas no disponibles en {path.name}: {missing}")
    usecols = [c for c in columns if c in available]
    return pd.read_csv(path, usecols=usecols, low_memory=False)

persona = read_existing(PERSONA_PATH, PERSONA_COLS)
hogar = read_existing(HOGAR_PATH, HOGAR_COLS)

for df in [persona, hogar]:
    for col in ["anio", "edad", WEIGHT, "factor_hogar", TARGET_MAIN, "ingreso_persona_total_registros_tri", TARGET_COMPLEMENT_PERSONA, "ingtrab_hogar_pc_oficial_tri", TARGET_COMPLEMENT_HOGAR, "ingtrab_pc_oficial_tri"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

persona_laboral = persona.loc[persona[TARGET_MAIN].gt(0)].copy()

shape_table = pd.DataFrame([
    {"base": "mart_persona", "filas": len(persona), "columnas_cargadas": persona.shape[1]},
    {"base": "mart_persona_laboral_positiva", "filas": len(persona_laboral), "columnas_cargadas": persona_laboral.shape[1]},
    {"base": "mart_hogar", "filas": len(hogar), "columnas_cargadas": hogar.shape[1]},
])
display(shape_table)

## Validación mínima

Antes de analizar, se revisa que las variables solicitadas existan, que no haya duplicados de llave y que la clasificación Banxico cubra las 32 entidades exactamente una vez.

In [ ]:
REQUESTED_COLS = [
    "region_banxico", "cve_ent", "entidad", "cve_mun", "municipio",
    "tam_loc", "tam_loc_desc", "est_socio", "est_socio_desc",
    "factor", "est_dis", "upm",
]

PERSON_KEY = ["anio", "folioviv", "foliohog", "numren"]
HOUSE_KEY = ["anio", "folioviv", "foliohog"]

def audit_mart(df, name, keys):
    rows = []
    for col in REQUESTED_COLS:
        exists = col in df.columns
        rows.append({
            "mart": name,
            "variable": col,
            "existe": exists,
            "faltantes": int(df[col].isna().sum()) if exists else None,
            "cobertura_pct": round(float(df[col].notna().mean() * 100), 4) if exists else None,
        })
    rows.append({
        "mart": name,
        "variable": "duplicados_llave",
        "existe": True,
        "faltantes": int(df.duplicated(keys).sum()),
        "cobertura_pct": None,
    })
    return pd.DataFrame(rows)

audit = pd.concat([
    audit_mart(persona, "mart_persona", PERSON_KEY),
    audit_mart(hogar, "mart_hogar", HOUSE_KEY),
], ignore_index=True)
display(audit)

entity_region = (
    pd.concat([
        persona[["cve_ent", "entidad", "region_banxico"]],
        hogar[["cve_ent", "entidad", "region_banxico"]],
    ], ignore_index=True)
    .drop_duplicates()
    .sort_values(["cve_ent", "entidad", "region_banxico"])
)
assert entity_region["cve_ent"].nunique() == 32, "La cobertura de entidades no es 32."
assert entity_region.duplicated(["cve_ent"]).sum() == 0, "Alguna entidad está asignada a más de una región."
assert entity_region["region_banxico"].isna().sum() == 0, "Hay entidades sin región Banxico."
display(entity_region)

## Target principal

El proyecto busca explicar diferencias en ingreso laboral. Por eso el target principal de esta libreta es `ingreso_persona_laboral_negocio_tri`. Para que las medianas sean interpretables, las tablas centrales usan la submuestra con ingreso laboral positivo. Como complemento territorial se conserva el ingreso corriente per cápita del hogar.

In [ ]:
def target_summary(df, target_cols):
    rows = []
    for col in target_cols:
        if col not in df.columns:
            continue
        s = pd.to_numeric(df[col], errors="coerce")
        rows.append({
            "target": col,
            "n_no_nulo": int(s.notna().sum()),
            "% ceros": round(float(s.fillna(0).eq(0).mean() * 100), 2),
            "media": s.mean(),
            "mediana": s.median(),
            "p75": s.quantile(0.75),
            "p95": s.quantile(0.95),
        })
    return pd.DataFrame(rows)

persona_targets = target_summary(persona, [
    TARGET_MAIN,
    "ingreso_persona_total_registros_tri",
    TARGET_COMPLEMENT_PERSONA,
    "ingtrab_hogar_pc_oficial_tri",
])
hogar_targets = target_summary(hogar, [TARGET_COMPLEMENT_HOGAR, "ingtrab_pc_oficial_tri"])

display(persona_targets)
display(hogar_targets)
print(f"Submuestra con ingreso laboral positivo: {len(persona_laboral):,} personas ({len(persona_laboral) / len(persona):.1%} del mart persona).")

## Funciones de resumen y visualización

Las siguientes funciones producen tablas descriptivas consistentes: tamaño de grupo, media, mediana, cuartiles y media ponderada descriptiva cuando existe `factor`.

In [ ]:
def money(x):
    if pd.isna(x):
        return ""
    return f"${x:,.0f}"

def compact_n(x):
    if pd.isna(x):
        return ""
    return f"{int(x):,}"

def weighted_mean(values, weights):
    values = pd.to_numeric(values, errors="coerce")
    weights = pd.to_numeric(weights, errors="coerce")
    mask = values.notna() & weights.notna() & weights.gt(0)
    if not mask.any():
        return np.nan
    return np.average(values[mask], weights=weights[mask])

def summary_table(df, by, target=TARGET_MAIN, weight=WEIGHT, min_n=0):
    by_cols = [by] if isinstance(by, str) else list(by)
    data = df[by_cols + [target] + ([weight] if weight in df.columns else [])].copy()
    for col in by_cols:
        data[col] = data[col].astype("object").where(data[col].notna(), "Sin dato / no aplica")
    data[target] = pd.to_numeric(data[target], errors="coerce")
    data = data.dropna(subset=[target])
    rows = []
    grouped = data.groupby(by_cols, dropna=False, sort=False)
    for key, group in grouped:
        key_tuple = key if isinstance(key, tuple) else (key,)
        s = group[target]
        row = dict(zip(by_cols, key_tuple))
        row.update({
            "n": int(s.size),
            "media": s.mean(),
            "media_pond": weighted_mean(s, group[weight]) if weight in group.columns else np.nan,
            "mediana": s.median(),
            "p25": s.quantile(0.25),
            "p75": s.quantile(0.75),
        })
        rows.append(row)
    out = pd.DataFrame(rows)
    if out.empty:
        return out
    out = out.loc[out["n"] >= min_n].copy()
    return out.sort_values("mediana", ascending=False).reset_index(drop=True)

def pretty_table(df):
    out = df.copy()
    for col in ["media", "media_pond", "mediana", "p25", "p75", "brecha_abs", "mediana_min", "mediana_max"]:
        if col in out.columns:
            out[col] = out[col].map(money)
    for col in ["n"]:
        if col in out.columns:
            out[col] = out[col].map(compact_n)
    for col in ["ratio_max_min"]:
        if col in out.columns:
            out[col] = out[col].map(lambda v: "" if pd.isna(v) else f"{v:,.2f}x")
    return out

def peso_axis(ax):
    ax.yaxis.set_major_formatter(FuncFormatter(lambda x, pos: f"${x:,.0f}"))
    return ax

def show_fig(fig):
    fig.tight_layout()
    display(fig)
    plt.close(fig)

def bar_median(table, x, title, top=None, order=None):
    data = table.copy()
    if top is not None:
        data = data.head(top)
    if order is not None:
        data[x] = pd.Categorical(data[x], categories=order, ordered=True)
        data = data.sort_values(x)
    fig, ax = plt.subplots(figsize=(10, 5))
    sns.barplot(data=data, x=x, y="mediana", ax=ax, color="#2F6B7A")
    peso_axis(ax)
    ax.set_title(title)
    ax.set_xlabel("")
    ax.set_ylabel("Mediana trimestral")
    ax.tick_params(axis="x", rotation=35)
    show_fig(fig)

## Distribución del ingreso laboral

Se revisa la distribución del target principal en la submuestra con ingreso laboral positivo. Para que la gráfica sea legible, el histograma se limita al percentil 99.

In [ ]:
cap = persona_laboral[TARGET_MAIN].quantile(0.99)
fig, ax = plt.subplots(figsize=(10, 5))
sns.histplot(persona_laboral.loc[persona_laboral[TARGET_MAIN].le(cap), TARGET_MAIN], bins=50, ax=ax, color="#2F6B7A")
peso_axis(ax)
ax.set_title("Distribución del ingreso laboral trimestral positivo")
ax.set_xlabel("Ingreso laboral trimestral")
ax.set_ylabel("Personas")
show_fig(fig)

overall = pd.DataFrame({
    "indicador": ["n", "media", "media_pond", "mediana", "p25", "p75", "p95"],
    "valor": [
        len(persona_laboral),
        persona_laboral[TARGET_MAIN].mean(),
        weighted_mean(persona_laboral[TARGET_MAIN], persona_laboral[WEIGHT]),
        persona_laboral[TARGET_MAIN].median(),
        persona_laboral[TARGET_MAIN].quantile(0.25),
        persona_laboral[TARGET_MAIN].quantile(0.75),
        persona_laboral[TARGET_MAIN].quantile(0.95),
    ]
})
overall["valor"] = overall.apply(lambda r: compact_n(r["valor"]) if r["indicador"] == "n" else money(r["valor"]), axis=1)
display(overall)

## Regiones Banxico

La primera lectura territorial usa cuatro regiones Banxico. Se muestran medianas, cuartiles y media ponderada descriptiva.

In [ ]:
region_stats = summary_table(persona_laboral, "region_banxico")
display(pretty_table(region_stats))
bar_median(region_stats, "region_banxico", "Ingreso laboral mediano por región Banxico", order=REGION_ORDER)

year_region = summary_table(persona_laboral, ["anio", "region_banxico"])
year_region["region_banxico"] = pd.Categorical(year_region["region_banxico"], categories=REGION_ORDER, ordered=True)
year_region = year_region.sort_values(["anio", "region_banxico"])
fig, ax = plt.subplots(figsize=(10, 5))
sns.lineplot(data=year_region, x="anio", y="mediana", hue="region_banxico", marker="o", ax=ax)
peso_axis(ax)
ax.set_title("Evolución del ingreso laboral mediano por región")
ax.set_xlabel("Año")
ax.set_ylabel("Mediana trimestral")
ax.legend(title="Región", loc="upper left")
show_fig(fig)

## Entidades federativas

El análisis estatal se mantiene descriptivo. No se usan mapas en esta etapa; se revisa la tabla completa de 32 entidades y una gráfica con los extremos de la distribución estatal.

In [ ]:
entity_stats = summary_table(persona_laboral, "entidad")
display(pretty_table(entity_stats))

extremes = pd.concat([entity_stats.head(8), entity_stats.tail(8)], ignore_index=True)
fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(data=extremes.sort_values("mediana"), y="entidad", x="mediana", ax=ax, color="#7A4E2F")
ax.xaxis.set_major_formatter(FuncFormatter(lambda x, pos: f"${x:,.0f}"))
ax.set_title("Entidades con menor y mayor ingreso laboral mediano")
ax.set_xlabel("Mediana trimestral")
ax.set_ylabel("")
show_fig(fig)

## Tamaño de localidad

Se usa `tam_loc_desc` tal como quedó documentado en los marts. Esta dimensión aproxima diferencias urbano-rurales sin añadir clasificaciones externas.

In [ ]:
tam_stats = summary_table(persona_laboral, "tam_loc_desc")
display(pretty_table(tam_stats))
bar_median(tam_stats, "tam_loc_desc", "Ingreso laboral mediano por tamaño de localidad")

year_tam = summary_table(persona_laboral, ["anio", "tam_loc_desc"])
fig, ax = plt.subplots(figsize=(10, 5))
sns.lineplot(data=year_tam, x="anio", y="mediana", hue="tam_loc_desc", marker="o", ax=ax)
peso_axis(ax)
ax.set_title("Evolución por tamaño de localidad")
ax.set_xlabel("Año")
ax.set_ylabel("Mediana trimestral")
ax.legend(title="Tamaño de localidad", bbox_to_anchor=(1.02, 1), loc="upper left")
show_fig(fig)

## Estrato socioeconómico

`est_socio_desc` se normalizó por código 1-4 para corregir una etiqueta dañada de 2022. No se modificó el código original `est_socio`.

In [ ]:
socio_order = ["Bajo", "Medio bajo", "Medio alto", "Alto"]
socio_stats = summary_table(persona_laboral, "est_socio_desc")
display(pretty_table(socio_stats))
bar_median(socio_stats, "est_socio_desc", "Ingreso laboral mediano por estrato socioeconómico", order=socio_order)

year_socio = summary_table(persona_laboral, ["anio", "est_socio_desc"])
year_socio["est_socio_desc"] = pd.Categorical(year_socio["est_socio_desc"], categories=socio_order, ordered=True)
year_socio = year_socio.sort_values(["anio", "est_socio_desc"])
fig, ax = plt.subplots(figsize=(10, 5))
sns.lineplot(data=year_socio, x="anio", y="mediana", hue="est_socio_desc", marker="o", ax=ax)
peso_axis(ax)
ax.set_title("Evolución por estrato socioeconómico")
ax.set_xlabel("Año")
ax.set_ylabel("Mediana trimestral")
ax.legend(title="Estrato", loc="upper left")
show_fig(fig)

## Educación por región

Se cruza `nivelaprob_desc` con región Banxico para revisar si el gradiente educativo es similar en todo el país. Las celdas con muy pocas observaciones pueden ser inestables, por lo que el resumen usa un mínimo de 100 personas por grupo.

In [ ]:
edu_region = summary_table(persona_laboral, ["nivelaprob_desc", "region_banxico"], min_n=100)
display(pretty_table(edu_region.head(40)))

pivot_edu = edu_region.pivot_table(index="nivelaprob_desc", columns="region_banxico", values="mediana", aggfunc="first")
order_edu = pivot_edu.median(axis=1).sort_values(ascending=False).index
pivot_edu = pivot_edu.loc[order_edu, [c for c in REGION_ORDER if c in pivot_edu.columns]]
fig, ax = plt.subplots(figsize=(10, 7))
sns.heatmap(pivot_edu, annot=True, fmt=".0f", cmap="YlGnBu", linewidths=0.5, ax=ax)
ax.set_title("Mediana de ingreso laboral por educación y región")
ax.set_xlabel("Región")
ax.set_ylabel("Nivel aprobado")
show_fig(fig)

## Sexo por región

Esta tabla no controla por edad, educación ni ocupación; por ahora solo permite ubicar si existe una brecha descriptiva persistente dentro de cada región.

In [ ]:
sexo_region = summary_table(persona_laboral, ["region_banxico", "sexo_desc"])
display(pretty_table(sexo_region))

pivot_sexo = sexo_region.pivot_table(index="region_banxico", columns="sexo_desc", values="mediana", aggfunc="first")
if {"Hombre", "Mujer"}.issubset(pivot_sexo.columns):
    pivot_sexo["brecha_hombre_mujer"] = pivot_sexo["Hombre"] - pivot_sexo["Mujer"]
    pivot_sexo["ratio_hombre_mujer"] = pivot_sexo["Hombre"] / pivot_sexo["Mujer"]
    display(pivot_sexo.reset_index())

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=sexo_region, x="region_banxico", y="mediana", hue="sexo_desc", order=REGION_ORDER, ax=ax)
peso_axis(ax)
ax.set_title("Ingreso laboral mediano por sexo y región")
ax.set_xlabel("Región")
ax.set_ylabel("Mediana trimestral")
ax.legend(title="Sexo")
show_fig(fig)

## Variables laborales

Se revisan condiciones laborales disponibles en el mart persona: subordinación, contrato, forma de pago y tamaño de empresa. Esto ayuda a formular hipótesis de modelado sin estimar todavía un modelo.

In [ ]:
labor_vars = [
    "subor_principal_desc",
    "contrato_principal_desc",
    "pago_principal_desc",
    "tam_emp_principal_desc",
]
for var in labor_vars:
    tbl = summary_table(persona_laboral, var, min_n=100)
    display(Markdown(f"### {var}"))
    display(pretty_table(tbl.head(20)))

contrato_region = summary_table(persona_laboral, ["region_banxico", "contrato_principal_desc"], min_n=100)
fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=contrato_region, x="region_banxico", y="mediana", hue="contrato_principal_desc", order=REGION_ORDER, ax=ax)
peso_axis(ax)
ax.set_title("Ingreso laboral mediano por contrato y región")
ax.set_xlabel("Región")
ax.set_ylabel("Mediana trimestral")
ax.legend(title="Contrato")
show_fig(fig)

## Ingreso corriente per cápita del hogar

Como contraste, se revisa el ingreso corriente per cápita a nivel hogar. Esta variable no reemplaza el target laboral principal, pero ayuda a distinguir brechas territoriales de bienestar del hogar.

In [ ]:
hogar_region = summary_table(hogar, "region_banxico", target=TARGET_COMPLEMENT_HOGAR)
hogar_tam = summary_table(hogar, "tam_loc_desc", target=TARGET_COMPLEMENT_HOGAR)
hogar_socio = summary_table(hogar, "est_socio_desc", target=TARGET_COMPLEMENT_HOGAR)

display(Markdown("### Hogar por región"))
display(pretty_table(hogar_region))
display(Markdown("### Hogar por tamaño de localidad"))
display(pretty_table(hogar_tam))
display(Markdown("### Hogar por estrato socioeconómico"))
display(pretty_table(hogar_socio))

bar_median(hogar_region, "region_banxico", "Ingreso corriente per cápita del hogar por región", order=REGION_ORDER)

## Asociaciones descriptivas

Para priorizar hipótesis se calcula, por variable categórica, la diferencia entre la mayor y menor mediana de ingreso laboral entre categorías con al menos 100 observaciones. Esto no implica causalidad; solo ordena posibles dimensiones explicativas.

In [ ]:
association_vars = [
    "region_banxico", "entidad", "tam_loc_desc", "est_socio_desc",
    "nivelaprob_desc", "sexo_desc", "subor_principal_desc",
    "contrato_principal_desc", "pago_principal_desc", "tam_emp_principal_desc",
]
rows = []
for var in association_vars:
    tbl = summary_table(persona_laboral, var, min_n=100)
    if tbl.empty:
        continue
    low = tbl.loc[tbl["mediana"].idxmin()]
    high = tbl.loc[tbl["mediana"].idxmax()]
    rows.append({
        "variable": var,
        "categoria_min": low[var],
        "mediana_min": low["mediana"],
        "categoria_max": high[var],
        "mediana_max": high["mediana"],
        "brecha_abs": high["mediana"] - low["mediana"],
        "ratio_max_min": high["mediana"] / low["mediana"] if low["mediana"] > 0 else np.nan,
        "n": int(tbl["n"].sum()),
    })
association_summary = pd.DataFrame(rows).sort_values("brecha_abs", ascending=False)
display(pretty_table(association_summary))

## Hallazgos preliminares

La siguiente celda genera hallazgos a partir de las tablas anteriores para que se actualicen si se vuelve a correr la libreta.

In [ ]:
def first_last(tbl, col="mediana"):
    high = tbl.loc[tbl[col].idxmax()]
    low = tbl.loc[tbl[col].idxmin()]
    return high, low

reg_hi, reg_lo = first_last(region_stats)
tam_hi, tam_lo = first_last(tam_stats)
socio_hi, socio_lo = first_last(socio_stats)
sexo_hi, sexo_lo = first_last(summary_table(persona_laboral, "sexo_desc"))
edu_tbl = summary_table(persona_laboral, "nivelaprob_desc", min_n=100)
edu_hi, edu_lo = first_last(edu_tbl)
hogar_hi, hogar_lo = first_last(hogar_region)
assoc_hi = association_summary.iloc[0]
last_year = int(persona_laboral["anio"].max())
first_year = int(persona_laboral["anio"].min())
first_last_region = year_region.pivot_table(index="region_banxico", columns="anio", values="mediana", aggfunc="first")
changes = (first_last_region[last_year] - first_last_region[first_year]).dropna().sort_values(ascending=False)

findings = [
    f"La muestra central tiene {len(persona_laboral):,} personas con ingreso laboral positivo, equivalente a {len(persona_laboral) / len(persona):.1%} del mart persona.",
    f"La región con mayor mediana de ingreso laboral es {reg_hi['region_banxico']} ({money(reg_hi['mediana'])}) y la menor es {reg_lo['region_banxico']} ({money(reg_lo['mediana'])}).",
    f"La brecha regional Norte-Sur aparece tanto en mediana laboral como en ingreso corriente per cápita del hogar: en hogares, {hogar_hi['region_banxico']} tiene la mediana más alta ({money(hogar_hi['mediana'])}) y {hogar_lo['region_banxico']} la más baja ({money(hogar_lo['mediana'])}).",
    f"El tamaño de localidad muestra un gradiente urbano-rural: {tam_hi['tam_loc_desc']} tiene mediana {money(tam_hi['mediana'])}, frente a {money(tam_lo['mediana'])} en {tam_lo['tam_loc_desc']}.",
    f"El estrato socioeconómico presenta una separación amplia: {socio_hi['est_socio_desc']} alcanza {money(socio_hi['mediana'])} y {socio_lo['est_socio_desc']} {money(socio_lo['mediana'])}.",
    f"La educación concentra una de las mayores diferencias descriptivas: {edu_hi['nivelaprob_desc']} registra {money(edu_hi['mediana'])}, mientras {edu_lo['nivelaprob_desc']} registra {money(edu_lo['mediana'])} entre categorías con al menos 100 casos.",
    f"La brecha por sexo es visible sin controles: {sexo_hi['sexo_desc']} tiene mediana {money(sexo_hi['mediana'])} y {sexo_lo['sexo_desc']} {money(sexo_lo['mediana'])}.",
    f"La variable con mayor brecha descriptiva simple es {assoc_hi['variable']}: {assoc_hi['categoria_max']} frente a {assoc_hi['categoria_min']}.",
    f"Entre {first_year} y {last_year}, la mediana regional sube en todas las regiones observadas; el mayor aumento absoluto se observa en {changes.index[0]} ({money(changes.iloc[0])}).",
    "Estos patrones son descriptivos y todavía no separan composición educativa, ocupacional, sexo, edad ni estructura del hogar; por eso deben leerse como insumo para hipótesis, no como explicación causal.",
]

display(Markdown("\n".join([f"- {item}" for item in findings])))

## Hipótesis para modelado

Estas hipótesis salen de la exploración descriptiva y pueden guiar la siguiente etapa. Todavía no son resultados de un modelo.

In [ ]:
hypotheses = [
    "La región Banxico conserva poder explicativo sobre el ingreso laboral aun después de controlar por educación, sexo, edad y variables laborales.",
    "Parte de la brecha Norte-Sur se explica por composición laboral: contrato, subordinación, forma de pago y tamaño de empresa.",
    "El tamaño de localidad captura una brecha urbano-rural que no se reduce completamente al estrato socioeconómico.",
    "El retorno descriptivo de la educación superior varía por región; por eso conviene probar interacciones entre educación y región.",
    "La brecha por sexo persiste dentro de regiones y debe evaluarse controlando educación, edad y tipo de inserción laboral.",
    "El estrato socioeconómico puede actuar como proxy de condiciones territoriales y de hogar, pero debe usarse con cuidado para no duplicar información del ingreso.",
    "Las diferencias estatales sugieren heterogeneidad dentro de cada región Banxico, especialmente en entidades turísticas, fronterizas e industriales.",
    "La evolución 2018-2024 sugiere cambios temporales relevantes; un modelo posterior debería incluir año y posiblemente interacciones año-región.",
]

display(Markdown("\n".join([f"- {item}" for item in hypotheses])))